# ERA5-Land Basic Energy Fluxes vs GEOS-LDAS OL/DA

This notebook compares GEOS-LDAS monthly flux output with ERA5-Land monthly averaged reanalysis on the M36 grid. The unit handling below is deliberately explicit and checked against the local NetCDF metadata before any regridding runs.

Documented ERA5-Land facts used here:

- Source product: `reanalysis-era5-land-monthly-means`, `product_type=monthly_averaged_reanalysis`, GRIB stream `moda`.
- The requested ERA5-Land fields are accumulated fields: `slhf` and `sshf` have delivered units `J m**-2`; `e` has delivered units `m of water equivalent`.
- For ERA5-Land monthly averaged data, accumulated variables in `moda` are monthly means of daily accumulations. Therefore `slhf`/`sshf` are interpreted as `J m-2 day-1`, and `e` as `m day-1`.
- ECMWF documents incorrect monthly averaged accumulated fields from September 2022 through February 2024. Because we do not yet have the documented workaround data (`mnth`, monthly averaged by hour at 00 UTC), the default below drops those months from metrics and figures, with the choice written into output metadata.

Model facts checked from the local flux files:

- `LHLAND` and `SHLAND` units are `W m-2`, with `cell_methods = time: mean`.
- `EVLAND` units are `kg m-2 s-1`, with `cell_methods = time: mean`.
- Monthly-total evaporation comparisons multiply model `EVLAND` by calendar-month seconds.

Sign convention for comparison: the GEOS-LDAS fields are positive for evaporation/latent heat from the land surface, checked from the local `LHLAND / EVLAND` latent-heat ratio. ERA5-Land accumulated turbulent and evaporation fields are multiplied by `-1` before comparison so positive values mean upward/from-surface fluxes in the comparison products.


In [ ]:
from pathlib import Path
import os
import sys
import gc
import struct
from collections import OrderedDict

os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/private/tmp")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xesmf as xe

xr.set_options(keep_attrs=True)
pd.options.display.max_columns = 120
pd.options.display.max_rows = 120


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / ".git").exists() and (p / "common/python/io/read_GEOSldas.py").exists():
            return p
    raise FileNotFoundError("Could not locate geosldas-analysis repo root")


HERE = Path.cwd().resolve()
REPO_ROOT = find_repo_root(HERE)
PROJECT_ROOT = REPO_ROOT / "projects/era5_land"
EASE_PATH = REPO_ROOT / "common/python/plotting/ease_grids"
OUT_DIR = PROJECT_ROOT / "output/energy_flux_comparison"
FIG_DIR = OUT_DIR / "figures"
REGRID_CACHE_DIR = OUT_DIR / "regridded_era5l_moda_dailyaccum_drop_202209_202402_cache"
MODEL_CACHE_DIR = OUT_DIR / "model_m36_cache"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
REGRID_CACHE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2")
TILECOORD = DATA_DIR / "LS_OLv8_M36.ldas_tilecoord.bin"
OL_FLUX_NC = DATA_DIR / "OLv8_flux_core_2000_2024_compressed.nc"
DA_FLUX_NC = DATA_DIR / "DAv8_flux_core_2000_2024_compressed.nc"
ERA5L_ENERGY_NC = PROJECT_ROOT / "data/era5l_monthly_energy_basics/era5l_monthly_energy_basics_2000_2025.nc"
WEIGHTS_PATH = PROJECT_ROOT / "weights_era5l_to_m36_consnormed.nc"

REGRID_BLOCK = 1
REGRID_DTYPE = "float32"
MIN_PAIRS = 24
RHO_WATER = 1000.0
ERA5L_SIGN_TO_UPWARD_POSITIVE = -1.0
SECONDS_PER_DAY = 86400.0

ERA5L_DOC_URL = "https://confluence.ecmwf.int/display/CKB/ERA5-Land%3A%2Bdata%2Bdocumentation"
ERA5L_PRODUCT = "reanalysis-era5-land-monthly-means"
ERA5L_PRODUCT_TYPE = "monthly_averaged_reanalysis"
ERA5L_STREAM = "moda"
DROP_ERA5L_KNOWN_BAD_ACCUM_MONTHS = True
ERA5L_KNOWN_BAD_ACCUM_START = "2022-09"
ERA5L_KNOWN_BAD_ACCUM_END = "2024-02"

OUTPUT_NC = OUT_DIR / "ERA5L_energy_fluxes_vs_OL_DA_M36_summary.nc"
OUTPUT_CSV = OUT_DIR / "ERA5L_energy_fluxes_vs_OL_DA_metric_summary.csv"
OUTPUT_PERIOD_CSV = OUT_DIR / "ERA5L_energy_fluxes_vs_OL_DA_periodized_metric_summary.csv"
OUTPUT_PERIOD_NC = OUT_DIR / "ERA5L_energy_fluxes_vs_OL_DA_periodized_M36_summary.nc"

PERIODS = [
    ("2000-06-01", "2007-05-31"),
    ("2007-06-01", "2015-03-31"),
    ("2015-04-01", "2024-05-31"),
]

for path in [TILECOORD, OL_FLUX_NC, DA_FLUX_NC, ERA5L_ENERGY_NC]:
    if not path.exists():
        raise FileNotFoundError(path)

print("REPO_ROOT:", REPO_ROOT)
print("ERA5-Land energy file:", ERA5L_ENERGY_NC)
print("Output:", OUTPUT_NC)


## Geometry Helpers

These are copied in spirit from the strict ERA5/ERA5-Land comparison workflow, but kept local so this notebook can evolve independently for flux diagnostics.

In [ ]:
def read_tilecoord(fname):
    """Read GEOS-LDAS tilecoord Fortran binary (little-endian)."""
    int_precision = "i"
    float_precision = "f"
    machfmt = "<"
    tile_coord = {}
    with open(fname, "rb") as ifp:
        _ = struct.unpack(f"{machfmt}i", ifp.read(4))[0]
        tile_coord["N_tile"] = struct.unpack(f"{machfmt}i", ifp.read(4))[0]
        _ = struct.unpack(f"{machfmt}i", ifp.read(4))[0]
        n_tile = tile_coord["N_tile"]
        fields = [
            "tile_id", "typ", "pfaf", "com_lon", "com_lat", "min_lon", "max_lon",
            "min_lat", "max_lat", "i_indg", "j_indg", "frac_cell", "frac_pfaf",
            "area", "elev",
        ]
        for field in fields:
            _ = struct.unpack(f"{machfmt}i", ifp.read(4))[0]
            dtype = int_precision if field in {"tile_id", "typ", "pfaf", "i_indg", "j_indg"} else float_precision
            arr = np.frombuffer(ifp.read(n_tile * 4), dtype=f"{machfmt}{dtype}")
            tile_coord[field] = arr.astype(np.float64 if dtype == "f" else np.int32)
            _ = struct.unpack(f"{machfmt}i", ifp.read(4))[0]
    return tile_coord


def build_m36_grid_cf(tc):
    """Build exact M36 grid centers/bounds from tilecoord cell bounds."""
    i_indg = tc["i_indg"].astype(int)
    j_indg = tc["j_indg"].astype(int)
    min_lon, max_lon = tc["min_lon"], tc["max_lon"]
    min_lat, max_lat = tc["min_lat"], tc["max_lat"]
    nx = int(i_indg.max()) + 1
    ny = int(j_indg.max()) + 1

    lon_b = np.full((ny + 1, nx + 1), np.nan)
    lat_b = np.full((ny + 1, nx + 1), np.nan)
    for lon0, lon1, lat0, lat1, i, j in zip(min_lon, max_lon, min_lat, max_lat, i_indg, j_indg):
        lon_b[j, i] = lon0
        lon_b[j, i + 1] = lon1
        lon_b[j + 1, i] = lon0
        lon_b[j + 1, i + 1] = lon1
        lat_b[j, i] = lat0
        lat_b[j + 1, i] = lat1
        lat_b[j, i + 1] = lat0
        lat_b[j + 1, i + 1] = lat1

    lon_c = 0.5 * (lon_b[:-1, :-1] + lon_b[1:, 1:])
    lat_c = 0.5 * (lat_b[:-1, :-1] + lat_b[1:, 1:])

    m36 = xr.Dataset(
        {
            "lon": (("y", "x"), lon_c),
            "lat": (("y", "x"), lat_c),
            "lon_b": (("y_b", "x_b"), lon_b),
            "lat_b": (("y_b", "x_b"), lat_b),
        }
    )
    m36["lon"].attrs.update(standard_name="longitude", units="degrees_east", bounds="lon_b")
    m36["lat"].attrs.update(standard_name="latitude", units="degrees_north", bounds="lat_b")
    m36["lon_b"].attrs.update(standard_name="longitude", units="degrees_east")
    m36["lat_b"].attrs.update(standard_name="latitude", units="degrees_north")
    return m36


def tiles_to_m36_grid(tile_values, tc, m36_grid, weights=None, land_only=True):
    """Map tile values to M36 cells using finite-tile weighted means."""
    ix = tc["i_indg"].astype(int)
    iy = tc["j_indg"].astype(int)
    ny = int(iy.max()) + 1
    nx = int(ix.max()) + 1

    tv = np.asarray(tile_values)
    if tv.ndim == 1:
        tv = tv[np.newaxis, :]
    n_time, n_tile = tv.shape
    if n_tile != ix.size:
        raise ValueError(f"tile dimension {n_tile} does not match tilecoord length {ix.size}")

    if weights is None:
        w = np.ones(n_tile, dtype=np.float64)
    else:
        w = np.asarray(weights, dtype=np.float64)
        if w.size != n_tile:
            raise ValueError(f"weights length {w.size} does not match tile dimension {n_tile}")

    sum_tyx = np.zeros((n_time, ny, nx), dtype=np.float64)
    wsum_tyx = np.zeros((n_time, ny, nx), dtype=np.float64)
    for t in range(n_time):
        v = tv[t]
        ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
        if not np.any(ok):
            continue
        np.add.at(sum_tyx[t], (iy[ok], ix[ok]), v[ok] * w[ok])
        np.add.at(wsum_tyx[t], (iy[ok], ix[ok]), w[ok])

    out = sum_tyx / np.where(wsum_tyx > 0, wsum_tyx, np.nan)
    da = xr.DataArray(
        out,
        dims=("time", "y", "x"),
        coords={
            "time": np.arange(n_time),
            "lat": (("y", "x"), m36_grid["lat"].values),
            "lon": (("y", "x"), m36_grid["lon"].values),
        },
        attrs={"note": "weighted mean of finite tiles per M36 cell"},
    )
    if land_only:
        da = da.where(np.isfinite(out).any(axis=0))
    return da



def tiles_to_m36_sum(tile_values, tc, m36_grid):
    """Sum finite tile values into M36 cells."""
    ix = tc["i_indg"].astype(int)
    iy = tc["j_indg"].astype(int)
    ny = int(iy.max()) + 1
    nx = int(ix.max()) + 1
    tv = np.asarray(tile_values, dtype=np.float64)
    if tv.size != ix.size:
        raise ValueError(f"tile_values length {tv.size} does not match tilecoord length {ix.size}")
    out = np.zeros((ny, nx), dtype=np.float64)
    ok = np.isfinite(tv)
    np.add.at(out, (iy[ok], ix[ok]), tv[ok])
    out = np.where(out > 0, out, np.nan)
    return xr.DataArray(
        out,
        dims=("y", "x"),
        coords={
            "lat": (("y", "x"), m36_grid["lat"].values),
            "lon": (("y", "x"), m36_grid["lon"].values),
        },
        attrs={"long_name": "summed tile land area per M36 cell", "units": "m2"},
    )


tc = read_tilecoord(TILECOORD)
m36_grid = build_m36_grid_cf(tc)
tile_area = np.asarray(tc["area"], dtype=np.float64)
cell_area = tiles_to_m36_sum(tile_area, tc, m36_grid).rename("cell_land_area")
print(f"N_tile = {tc['N_tile']}")
print("M36 grid:", dict(m36_grid.sizes))
print("tile area finite:", int(np.isfinite(tile_area).sum()))
print("M36 land-area cells:", int(np.isfinite(cell_area).sum()))

## Conversion and Metric Helpers

In [ ]:
def month_seconds(time_values):
    period = pd.PeriodIndex(pd.to_datetime(time_values), freq="M")
    starts = period.to_timestamp(how="start")
    ends = (period + 1).to_timestamp(how="start")
    seconds = (ends - starts).total_seconds().astype("float64")
    return xr.DataArray(seconds, dims="time", coords={"time": time_values}, name="seconds_per_month")



def month_days(time_values):
    days = month_seconds(time_values).values / SECONDS_PER_DAY
    return xr.DataArray(days, dims="time", coords={"time": time_values}, name="days_per_month")




def assert_attr_contains(obj, attr_name, expected, label):
    actual = obj.attrs.get(attr_name)
    if actual is None or expected not in str(actual):
        raise AssertionError(f"{label}: expected {attr_name} to contain {expected!r}; found {actual!r}")


def assert_attr_equal(obj, attr_name, expected, label):
    actual = obj.attrs.get(attr_name)
    if actual != expected:
        raise AssertionError(f"{label}: expected {attr_name}={expected!r}; found {actual!r}")


def assert_era5l_energy_metadata(ds):
    history = ds.attrs.get("history", "")
    if ERA5L_STREAM not in history:
        raise AssertionError(f"ERA5-Land file history does not show stream {ERA5L_STREAM!r}: {history[:500]!r}")

    expected = {
        "slhf": {"units": "J m**-2", "GRIB_units": "J m**-2", "GRIB_shortName": "slhf", "GRIB_dataType": "fc"},
        "sshf": {"units": "J m**-2", "GRIB_units": "J m**-2", "GRIB_shortName": "sshf", "GRIB_dataType": "fc"},
        "e": {"units": "m of water equivalent", "GRIB_units": "m of water equivalent", "GRIB_shortName": "e", "GRIB_dataType": "fc"},
    }
    for var, checks in expected.items():
        if var not in ds:
            raise AssertionError(f"ERA5-Land field {var!r} is missing")
        for attr, expected_value in checks.items():
            assert_attr_equal(ds[var], attr, expected_value, f"ERA5-Land {var}")


def assert_model_flux_metadata(ds, var):
    expected = {
        "LHLAND": {"units": "W m-2", "cell_methods": "time: mean"},
        "SHLAND": {"units": "W m-2", "cell_methods": "time: mean"},
        "EVLAND": {"units": "kg m-2 s-1", "cell_methods": "time: mean"},
    }
    if var not in expected:
        raise AssertionError(f"No model metadata contract defined for {var}")
    if var not in ds:
        raise AssertionError(f"Model field {var!r} is missing")
    for attr, expected_value in expected[var].items():
        assert_attr_equal(ds[var], attr, expected_value, f"model {var}")


def known_bad_era5l_accum_mask(time_values):
    months = to_month_period(time_values).astype(str)
    return (months >= ERA5L_KNOWN_BAD_ACCUM_START) & (months <= ERA5L_KNOWN_BAD_ACCUM_END)

def to_month_period(values):
    return pd.PeriodIndex(pd.to_datetime(values), freq="M")


def anomalies_monthly(da, dim="time"):
    clim = da.groupby(f"{dim}.month").mean(dim=dim, skipna=True)
    return da.groupby(f"{dim}.month") - clim


def metric_pair(model, ref, min_pairs=MIN_PAIRS):
    valid = xr.where(np.isfinite(model) & np.isfinite(ref), True, False)
    n = valid.sum("time")
    model_v = model.where(valid).astype("float32")
    ref_v = ref.where(valid).astype("float32")

    cov = ((model_v - model_v.mean("time", skipna=True)) * (ref_v - ref_v.mean("time", skipna=True))).mean("time", skipna=True)
    var_m = ((model_v - model_v.mean("time", skipna=True)) ** 2).mean("time", skipna=True)
    var_r = ((ref_v - ref_v.mean("time", skipna=True)) ** 2).mean("time", skipna=True)
    corr = xr.where((var_m > 0) & (var_r > 0), cov / np.sqrt(var_m * var_r), np.nan)

    model_an = anomalies_monthly(model_v)
    ref_an = anomalies_monthly(ref_v)
    an_valid = xr.where(np.isfinite(model_an) & np.isfinite(ref_an), True, False)
    an_n = an_valid.sum("time")
    model_an = model_an.where(an_valid)
    ref_an = ref_an.where(an_valid)
    an_cov = (model_an * ref_an).mean("time", skipna=True)
    an_var_m = (model_an ** 2).mean("time", skipna=True)
    an_var_r = (ref_an ** 2).mean("time", skipna=True)
    anom_corr = xr.where((an_var_m > 0) & (an_var_r > 0), an_cov / np.sqrt(an_var_m * an_var_r), np.nan)
    ubrmse = np.sqrt(((model_an - ref_an) ** 2).mean("time", skipna=True))

    diff = model_v - ref_v
    rmse = np.sqrt((diff ** 2).mean("time", skipna=True))
    bias = diff.mean("time", skipna=True)

    return xr.Dataset(
        {
            "R": xr.where(n >= min_pairs, corr, np.nan),
            "anomR": xr.where(an_n >= min_pairs, anom_corr, np.nan),
            "ubRMSE": xr.where(an_n >= min_pairs, ubrmse, np.nan),
            "RMSE": xr.where(n >= min_pairs, rmse, np.nan),
            "bias": xr.where(n >= min_pairs, bias, np.nan),
            "N": n,
        }
    )


def area_mean(da):
    weights = cell_area.where(np.isfinite(da))
    return da.weighted(weights.fillna(0)).mean(("y", "x"), skipna=True)


def period_label(t0, t1):
    return f"{t0}_to_{t1}"


def select_period(da, t0, t1):
    months = to_month_period(da.time.values)
    p0 = pd.Period(pd.Timestamp(t0), freq="M")
    p1 = pd.Period(pd.Timestamp(t1), freq="M")
    keep = np.where((months >= p0) & (months <= p1))[0]
    return da.isel(time=keep)


def append_period_metric_rows(rows, comparison, spec, period, ol_metrics, da_metrics, n_months):
    t0, t1 = period
    label = period_label(t0, t1)
    stats = ["R", "anomR", "ubRMSE", "RMSE", "bias", "N"]
    for exp, ds_metric in [("OL", ol_metrics), ("DA", da_metrics)]:
        for stat in stats:
            val = area_mean(ds_metric[stat]) if stat != "N" else ds_metric[stat].mean(("y", "x"), skipna=True)
            rows.append(
                {
                    "comparison": comparison,
                    "period": label,
                    "period_start": t0,
                    "period_end": t1,
                    "experiment": exp,
                    "stat": stat,
                    "spatial_mean": float(val.values),
                    "units": "count" if stat == "N" else ("1" if stat in {"R", "anomR"} else spec["units"]),
                    "n_months": int(n_months),
                }
            )

    for stat in ["R", "anomR", "ubRMSE", "RMSE", "bias"]:
        delta = da_metrics[stat] - ol_metrics[stat]
        val = area_mean(delta)
        rows.append(
            {
                "comparison": comparison,
                "period": label,
                "period_start": t0,
                "period_end": t1,
                "experiment": "DA_minus_OL",
                "stat": stat,
                "spatial_mean": float(val.values),
                "units": "1" if stat in {"R", "anomR"} else spec["units"],
                "n_months": int(n_months),
            }
        )


def robust_symmetric_norm(*arrays, percentile=98.0, floor=1.0e-8):
    vals = []
    for arr in arrays:
        a = np.asarray(arr)
        vals.append(a[np.isfinite(a)].ravel())
    finite = np.concatenate([v for v in vals if v.size]) if any(v.size for v in vals) else np.array([])
    vmax = np.nanpercentile(np.abs(finite), percentile) if finite.size else floor
    vmax = max(float(vmax), floor)
    return mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)


LAT_MIN_MAP = -60.0
EASE_SHAPE = (406, 964)
_ease_plot_grid_cache = None


def savefig(fig, stem):
    png = FIG_DIR / f"{stem}.png"
    fig.savefig(png, dpi=220, bbox_inches="tight")
    print("saved", png)


def ease_plot_grid(shape):
    global _ease_plot_grid_cache
    if _ease_plot_grid_cache is None:
        lat_path = EASE_PATH / "EASE2_M36km.lats.964x406x1.double"
        lon_path = EASE_PATH / "EASE2_M36km.lons.964x406x1.double"
        if not lat_path.exists() or not lon_path.exists():
            raise FileNotFoundError(f"Missing EASE plotting grid under {EASE_PATH}")
        lats2d = np.fromfile(lat_path, dtype=np.float64).reshape(EASE_SHAPE)
        lons2d = np.fromfile(lon_path, dtype=np.float64).reshape(EASE_SHAPE)
        _ease_plot_grid_cache = (lons2d, lats2d)
    lons2d, lats2d = _ease_plot_grid_cache
    ny, nx = shape
    return lons2d[:ny, :nx], lats2d[:ny, :nx]


def add_map_base(ax):
    ax.add_feature(cfeature.LAND, facecolor="0.92", edgecolor="none", zorder=0)
    ax.coastlines(resolution="50m", linewidth=0.45, color="0.35")
    ax.set_global()
    ax.set_extent([-180, 180, LAT_MIN_MAP, 90], crs=ccrs.PlateCarree())


def plot_robinson_m36_field(ax, field, *, cmap="RdBu_r", norm=None):
    values = np.asarray(field, dtype=float)
    lon_plot, lat_plot = ease_plot_grid(values.shape)
    values = np.where(lat_plot >= LAT_MIN_MAP, values, np.nan)
    mesh = ax.pcolormesh(
        lon_plot,
        lat_plot,
        np.ma.masked_invalid(values),
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        norm=norm,
        shading="auto",
        rasterized=True,
    )
    add_map_base(ax)
    return mesh


## Load and Regrid ERA5-Land

ERA5-Land metadata is validated before conversion. For `moda` monthly averaged accumulated fields, the documented interpretation is daily accumulation per day: `slhf`/`sshf` in `J m-2 day-1` and `e` in `m day-1`. The notebook converts those to model-comparable `W m-2`, `kg m-2 month-1` for the periodized/reporting comparison.

The documented September 2022-February 2024 issue for monthly averaged accumulated fields is handled explicitly by `DROP_ERA5L_KNOWN_BAD_ACCUM_MONTHS`.


In [ ]:
ds_era = xr.open_dataset(ERA5L_ENERGY_NC)
assert_era5l_energy_metadata(ds_era)
print("Validated ERA5-Land accumulated-field metadata against local NetCDF attributes")
for req in ["valid_time", "latitude", "longitude", "slhf", "sshf", "e"]:
    if req not in ds_era.variables and req not in ds_era.coords and req not in ds_era.dims:
        raise RuntimeError(f"Missing required ERA5-Land field: {req}")

ds_era = ds_era.rename({"valid_time": "time"})
if ds_era.latitude[0] > ds_era.latitude[-1]:
    ds_era = ds_era.reindex(latitude=list(reversed(ds_era.latitude.values)))
ds_era = ds_era.assign_coords(longitude=np.mod(ds_era.longitude, 360.0)).sortby("longitude")

# Regrid only months overlapping the model flux files. This cuts peak work and output volume.
with xr.open_dataset(OL_FLUX_NC, decode_times=True) as _model_time_ds:
    _model_months = to_month_period(_model_time_ds.time.values).astype(str)
_era_months = to_month_period(ds_era.time.values).astype(str)
_keep_mask = np.isin(_era_months, _model_months)
_known_bad_mask = known_bad_era5l_accum_mask(ds_era.time.values)
known_bad_overlap = _era_months[_keep_mask & _known_bad_mask]
if known_bad_overlap.size:
    print(
        "ERA5-Land documented accumulated-field issue months in model overlap:",
        known_bad_overlap[0], "to", known_bad_overlap[-1], f"({known_bad_overlap.size} months)",
    )
if DROP_ERA5L_KNOWN_BAD_ACCUM_MONTHS:
    _keep_mask = _keep_mask & (~_known_bad_mask)
_keep = np.where(_keep_mask)[0]
if _keep.size == 0:
    raise RuntimeError("No overlapping model/ERA5-Land months before regridding after documented-quality filters")
ds_era = ds_era.isel(time=_keep)
print(
    "ERA5-Land months selected for regrid:",
    ds_era.sizes["time"],
    str(ds_era.time.values[0])[:10], "to", str(ds_era.time.values[-1])[:10],
    "drop_known_bad_accum_months=", DROP_ERA5L_KNOWN_BAD_ACCUM_MONTHS,
)

days = month_days(ds_era.time.values)
# ERA5-Land moda monthly means of accumulated turbulent fluxes: J m-2 day-1 -> W m-2.
era_lh_native = (ERA5L_SIGN_TO_UPWARD_POSITIVE * ds_era["slhf"] / SECONDS_PER_DAY).rename("ERA5L_LHLAND")
era_sh_native = (ERA5L_SIGN_TO_UPWARD_POSITIVE * ds_era["sshf"] / SECONDS_PER_DAY).rename("ERA5L_SHLAND")
# ERA5-Land moda evaporation: m day-1 -> kg m-2 s-1 and kg m-2 month-1.
era_et_daily_native = (ERA5L_SIGN_TO_UPWARD_POSITIVE * ds_era["e"] * RHO_WATER).rename("ERA5L_EVLAND_daily")
era_et_rate_native = (era_et_daily_native / SECONDS_PER_DAY).rename("ERA5L_EVLAND")
era_et_month_native = (era_et_daily_native * days).rename("ERA5L_EVLAND_month")

weights_exist = WEIGHTS_PATH.exists()
regridder = xe.Regridder(
    ds_era.isel(time=slice(0, 1)),
    m36_grid,
    method="conservative_normed",
    periodic=True,
    filename=str(WEIGHTS_PATH),
    reuse_weights=weights_exist,
)
print(regridder)
print(f"Weights {'reused' if weights_exist else 'created'}: {WEIGHTS_PATH}")


def regrid_time_blocks(da, name, block=REGRID_BLOCK, dtype=REGRID_DTYPE, valid_min=None, valid_max=None):
    cache_file = REGRID_CACHE_DIR / f"{name}_m36.nc"
    if cache_file.exists():
        print(f"Loading cached regrid for {name}: {cache_file}")
        with xr.open_dataset(cache_file) as cached:
            return cached[name].load()

    parts = []
    nt = da.sizes["time"]
    mv = da.attrs.get("GRIB_missingValue", None)
    mv_thresh = (0.5 * float(mv)) if ((mv is not None) and np.isfinite(mv)) else None
    for start in range(0, nt, block):
        stop = min(start + block, nt)
        print(f"Regridding {name}: {start}:{stop} / {nt}", flush=True)
        blk = da.isel(time=slice(start, stop)).astype(dtype)
        if mv_thresh is not None:
            blk = blk.where(blk < mv_thresh)
        if valid_min is not None:
            blk = blk.where(blk >= valid_min)
        if valid_max is not None:
            blk = blk.where(blk <= valid_max)
        part = regridder(blk).rename(name)
        if valid_min is not None:
            part = part.where(part >= valid_min)
        if valid_max is not None:
            part = part.where(part <= valid_max)
        part = part.where(np.isfinite(part)).astype(dtype).load()
        parts.append(part)
        del blk, part
        gc.collect()
    out = xr.concat(parts, dim="time").rename(name)
    out.to_dataset(name=name).to_netcdf(cache_file, mode="w")
    print(f"Saved cached regrid for {name}: {cache_file}")
    del parts
    gc.collect()
    return out


era_lh = regrid_time_blocks(era_lh_native, "ERA5L_LHLAND", valid_min=-500.0, valid_max=500.0)
era_sh = regrid_time_blocks(era_sh_native, "ERA5L_SHLAND", valid_min=-500.0, valid_max=500.0)
era_et = regrid_time_blocks(era_et_rate_native, "ERA5L_EVLAND", valid_min=-0.01, valid_max=0.01)
era_et_month = regrid_time_blocks(era_et_month_native, "ERA5L_EVLAND_month", valid_min=-500.0, valid_max=500.0)

try:
    ds_era.close()
except Exception:
    pass

print("ERA5-Land regridded:", era_lh.shape, era_sh.shape, era_et.shape)

## Load Model Fluxes and Compute One Comparison at a Time

The ERA5-Land regrids are cached above. To keep peak memory low, the next cells load one reference variable and one OL/DA model variable at a time, compute metrics and domain time series, then release the monthly fields before moving to the next comparison.

In [ ]:
def _dataset_month_strings(path):
    with xr.open_dataset(path, decode_times=True) as ds:
        return to_month_period(ds.time.values).astype(str), ds.time.values


def load_cached_ref(name):
    cache_file = REGRID_CACHE_DIR / f"{name}_m36.nc"
    if not cache_file.exists():
        raise FileNotFoundError(cache_file)
    with xr.open_dataset(cache_file) as ds:
        return ds[name].load()


def map_tiles_to_m36_grid_blocks(tile_values, times, name, block=12):
    parts = []
    n_time = tile_values.shape[0]
    for start in range(0, n_time, block):
        stop = min(start + block, n_time)
        part = tiles_to_m36_grid(
            tile_values[start:stop].astype("float64", copy=False),
            tc,
            m36_grid,
            weights=tile_area,
            land_only=True,
        )
        part = part.assign_coords(time=times[start:stop]).rename(name).astype("float32").load()
        parts.append(part)
        del part
        gc.collect()
    out = xr.concat(parts, dim="time").rename(name)
    del parts
    gc.collect()
    return out


def load_model_m36(path, label, var, out_name, monthly_total=False):
    suffix = "month" if monthly_total else "rate"
    cache_file = MODEL_CACHE_DIR / f"{label}_{out_name}_{suffix}_m36.nc"
    if cache_file.exists():
        print(f"Loading cached model grid for {label} {out_name}: {cache_file}")
        with xr.open_dataset(cache_file) as cached:
            return cached[f"{label}_{out_name}"].load()

    with xr.open_dataset(path, decode_times=True) as ds:
        assert_model_flux_metadata(ds, var)
        values = ds[var].values.astype("float32")
        times = ds.time.values
        if monthly_total:
            sec = month_seconds(times).values.astype("float32")
            values = values * sec[:, None]
        out = map_tiles_to_m36_grid_blocks(values, times, f"{label}_{out_name}", block=12)
        out.attrs.update(ds[var].attrs)
        if monthly_total:
            out.attrs.update(long_name="Total evaporation monthly total", units="kg m-2 month-1")
        del values
        gc.collect()

    out.to_dataset(name=out.name).to_netcdf(cache_file, mode="w")
    print(f"Saved cached model grid for {label} {out_name}: {cache_file}")
    return out


def align_triplet(ol, da, ref):
    model_months = to_month_period(ol.time.values).astype(str)
    da_months = to_month_period(da.time.values).astype(str)
    ref_months = to_month_period(ref.time.values).astype(str)
    common = np.intersect1d(np.intersect1d(model_months, da_months), ref_months)
    if common.size == 0:
        raise RuntimeError("No common months among OL, DA, and ERA5-Land")
    common_time = pd.PeriodIndex(common, freq="M").to_timestamp(how="end")
    ol_i = np.where(np.isin(model_months, common))[0]
    da_i = np.where(np.isin(da_months, common))[0]
    ref_i = np.where(np.isin(ref_months, common))[0]
    return (
        ol.isel(time=ol_i).assign_coords(time=common_time),
        da.isel(time=da_i).assign_coords(time=common_time),
        ref.isel(time=ref_i).assign_coords(time=common_time),
    )

## Metrics and Summary Tables

In [ ]:
COMPARISONS = OrderedDict(
    [
        ("latent_heat", {"label": "latent heat", "model_var": "LHLAND", "model_out": "LHLAND", "ref": "ERA5L_LHLAND", "units": "W m-2", "monthly_total": False}),
        ("sensible_heat", {"label": "sensible heat", "model_var": "SHLAND", "model_out": "SHLAND", "ref": "ERA5L_SHLAND", "units": "W m-2", "monthly_total": False}),
        ("evaporation", {"label": "evaporation", "model_var": "EVLAND", "model_out": "EVLAND_month", "ref": "ERA5L_EVLAND_month", "units": "kg m-2 month-1", "monthly_total": True}),
    ]
)

metric_vars = {}
period_metric_vars = {}
summary_rows = []
period_summary_rows = []
timeseries_rows = []

period_labels = [period_label(t0, t1) for t0, t1 in PERIODS]
period_starts = [t0 for t0, _ in PERIODS]
period_ends = [t1 for _, t1 in PERIODS]

for comparison, spec in COMPARISONS.items():
    print(f"\n=== {comparison} ===", flush=True)
    ref = load_cached_ref(spec["ref"])
    ol = load_model_m36(OL_FLUX_NC, "OL", spec["model_var"], spec["model_out"], monthly_total=spec["monthly_total"])
    da = load_model_m36(DA_FLUX_NC, "DA", spec["model_var"], spec["model_out"], monthly_total=spec["monthly_total"])
    ol, da, ref = align_triplet(ol, da, ref)
    print("Aligned months:", ol.sizes["time"], str(ol.time.values[0])[:10], "to", str(ol.time.values[-1])[:10])

    ol_metrics = metric_pair(ol, ref)
    da_metrics = metric_pair(da, ref)
    for exp, ds_metric in [("OL", ol_metrics), ("DA", da_metrics)]:
        for stat in ["R", "anomR", "ubRMSE", "RMSE", "bias", "N"]:
            out_name = f"{exp}_{comparison}_{stat}"
            metric_vars[out_name] = ds_metric[stat].astype("float32")
            val = area_mean(ds_metric[stat]) if stat != "N" else ds_metric[stat].mean(("y", "x"), skipna=True)
            summary_rows.append(
                {
                    "comparison": comparison,
                    "experiment": exp,
                    "stat": stat,
                    "spatial_mean": float(val.values),
                    "units": "count" if stat == "N" else ("1" if stat in {"R", "anomR"} else spec["units"]),
                }
            )
    for exp, field in [("ERA5L", ref), ("OL", ol), ("DA", da)]:
        ts = area_mean(field).load()
        for t, val in zip(pd.to_datetime(ts.time.values), ts.values):
            timeseries_rows.append(
                {
                    "time": t.date().isoformat(),
                    "comparison": comparison,
                    "experiment": exp,
                    "value": float(val) if np.isfinite(val) else np.nan,
                    "units": spec["units"],
                }
            )

    for period in PERIODS:
        t0, t1 = period
        label = period_label(t0, t1)
        ol_per = select_period(ol, t0, t1)
        da_per = select_period(da, t0, t1)
        ref_per = select_period(ref, t0, t1)
        n_months = ol_per.sizes.get("time", 0)
        if n_months < MIN_PAIRS:
            print(f"Skipping {comparison} {label}: only {n_months} months")
            continue
        ol_per_metrics = metric_pair(ol_per, ref_per)
        da_per_metrics = metric_pair(da_per, ref_per)
        append_period_metric_rows(period_summary_rows, comparison, spec, period, ol_per_metrics, da_per_metrics, n_months)

        for exp, ds_metric in [("OL", ol_per_metrics), ("DA", da_per_metrics)]:
            for stat in ["R", "anomR", "ubRMSE", "RMSE", "bias", "N"]:
                name = f"{exp}_{comparison}_{stat}"
                period_metric_vars.setdefault(name, []).append(ds_metric[stat].astype("float32"))
        for stat in ["R", "anomR", "ubRMSE", "RMSE", "bias"]:
            name = f"DA_minus_OL_{comparison}_{stat}"
            period_metric_vars.setdefault(name, []).append((da_per_metrics[stat] - ol_per_metrics[stat]).astype("float32"))

        del ol_per, da_per, ref_per, ol_per_metrics, da_per_metrics
        gc.collect()

    del ref, ol, da, ol_metrics, da_metrics
    gc.collect()

metric_ds = xr.Dataset(metric_vars).assign_coords(
    lat=cell_area["lat"],
    lon=cell_area["lon"],
    cell_land_area=cell_area,
)
period_metric_ds = xr.Dataset(
    {name: xr.concat(arrs, dim=xr.IndexVariable("period", period_labels)).astype("float32") for name, arrs in period_metric_vars.items()}
).assign_coords(
    period_start=("period", period_starts),
    period_end=("period", period_ends),
    lat=cell_area["lat"],
    lon=cell_area["lon"],
    cell_land_area=cell_area,
)
metric_summary = pd.DataFrame(summary_rows)
period_metric_summary = pd.DataFrame(period_summary_rows)
domain_timeseries = pd.DataFrame(timeseries_rows)
metric_summary.to_csv(OUTPUT_CSV, index=False)
period_metric_summary.to_csv(OUTPUT_PERIOD_CSV, index=False)
domain_timeseries.to_csv(OUT_DIR / "ERA5L_energy_fluxes_vs_OL_DA_domain_timeseries.csv", index=False)
display(metric_summary)
display(period_metric_summary.head(36))
print("saved", OUTPUT_CSV)
print("saved", OUTPUT_PERIOD_CSV)
print("saved", OUT_DIR / "ERA5L_energy_fluxes_vs_OL_DA_domain_timeseries.csv")
print("period_metric_ds variables:", len(period_metric_ds.data_vars), "periods:", list(period_metric_ds.period.values))


## Periodized Energy Figures

These figures follow the snow/SM periodized summaries: metrics are computed at each M36 grid cell within each period, then summarized for bars or mapped as DA-minus-OL grid-cell changes.


In [ ]:
if period_metric_summary.empty:
    raise RuntimeError("period_metric_summary is empty; run the metric cell first")

period_display = [f"{pd.Timestamp(t0).strftime('%Y-%m')} to {pd.Timestamp(t1).strftime('%Y-%m')}" for t0, t1 in PERIODS]
exp_colors = {"OL": "#4c78a8", "DA": "#f58518"}

# Bar chart: rows are variables, columns are key metrics, bars are OL/DA within each period.
bar_stats = ["R", "anomR", "RMSE", "bias"]
fig, axes = plt.subplots(len(COMPARISONS), len(bar_stats), figsize=(15, 10), constrained_layout=True)
for r, (comparison, spec) in enumerate(COMPARISONS.items()):
    for c, stat in enumerate(bar_stats):
        ax = axes[r, c]
        sub = period_metric_summary[
            (period_metric_summary["comparison"] == comparison)
            & (period_metric_summary["stat"] == stat)
            & (period_metric_summary["experiment"].isin(["OL", "DA"]))
        ]
        x = np.arange(len(PERIODS), dtype=float)
        width = 0.36
        for offset, exp in [(-width / 2, "OL"), (width / 2, "DA")]:
            vals = []
            for label in period_labels:
                row = sub[(sub["period"] == label) & (sub["experiment"] == exp)]
                vals.append(float(row["spatial_mean"].iloc[0]) if len(row) else np.nan)
            ax.bar(x + offset, vals, width=width, color=exp_colors[exp], edgecolor="black", linewidth=0.5, label=exp if (r == 0 and c == 0) else None)
        ax.set_xticks(x)
        ax.set_xticklabels([f"P{i+1}" for i in range(len(PERIODS))])
        if r == 0:
            ax.set_title(stat)
        if c == 0:
            ax.set_ylabel(spec["label"])
        if stat in {"R", "anomR"}:
            ax.set_ylim(0.0, 1.0)
        ax.grid(True, axis="y", alpha=0.25)
fig.legend(loc="upper center", ncol=2, frameon=False)
fig.suptitle("ERA5-Land energy metrics by period: grid-cell metrics, spatially summarized", y=1.02)
savefig(fig, "era5l_energy_flux_periodized_bars")
plt.show()

# Postage maps: DA minus OL changes in anomaly correlation and bias, by period.
def plot_periodized_delta_maps(stat, stem, title, unit_by_comparison):
    fig, axes = plt.subplots(
        len(COMPARISONS),
        len(PERIODS),
        figsize=(13.8, 8.8),
        subplot_kw={"projection": ccrs.Robinson()},
        constrained_layout=False,
    )
    axes = np.atleast_2d(axes)
    plt.subplots_adjust(left=0.075, right=0.895, top=0.91, bottom=0.055, wspace=0.02, hspace=0.06)

    for r, (comparison, spec) in enumerate(COMPARISONS.items()):
        vals = period_metric_ds[f"DA_minus_OL_{comparison}_{stat}"]
        norm = robust_symmetric_norm(*[vals.sel(period=label).values for label in period_labels], percentile=98.0)
        artist = None
        for c, label in enumerate(period_labels):
            ax = axes[r, c]
            z = vals.sel(period=label).values
            artist = plot_robinson_m36_field(ax, z, cmap="RdBu_r", norm=norm)
            if r == 0:
                ax.set_title(f"P{c+1}\n{period_display[c]}", fontsize=9, pad=3)
            if c == 0:
                ax.text(
                    -0.08,
                    0.5,
                    spec["label"],
                    transform=ax.transAxes,
                    ha="right",
                    va="center",
                    rotation=90,
                    fontsize=10,
                    weight="bold",
                )
        cb = fig.colorbar(artist, ax=axes[r, :].tolist(), orientation="vertical", fraction=0.022, pad=0.012)
        cb.set_label(unit_by_comparison(comparison, spec))
    fig.suptitle(title, y=0.975, fontsize=13)
    savefig(fig, stem)
    plt.show()

plot_periodized_delta_maps(
    "anomR",
    "era5l_energy_flux_periodized_da_minus_ol_anomR_maps",
    "DA minus OL anomaly-correlation change by period",
    lambda comparison, spec: "DA - OL anomR (1)",
)
plot_periodized_delta_maps(
    "bias",
    "era5l_energy_flux_periodized_da_minus_ol_bias_maps",
    "DA minus OL bias change by period",
    lambda comparison, spec: spec["units"],
)


## Quick Figures

In [ ]:
fig, axes = plt.subplots(
    len(COMPARISONS),
    3,
    figsize=(13.8, 8.8),
    subplot_kw={"projection": ccrs.Robinson()},
    constrained_layout=False,
)
axes = np.atleast_2d(axes)
plt.subplots_adjust(left=0.075, right=0.895, top=0.91, bottom=0.055, wspace=0.02, hspace=0.06)

for row, (metric_name, spec) in enumerate(COMPARISONS.items()):
    ol_bias = metric_ds[f"OL_{metric_name}_bias"]
    da_bias = metric_ds[f"DA_{metric_name}_bias"]
    delta_bias = da_bias - ol_bias
    norm = robust_symmetric_norm(ol_bias, da_bias, delta_bias, percentile=98.0)
    artist = None
    for col, (title, field) in enumerate([
        ("OL - ERA5L", ol_bias),
        ("DA - ERA5L", da_bias),
        ("DA bias - OL bias", delta_bias),
    ]):
        ax = axes[row, col]
        artist = plot_robinson_m36_field(ax, field.values, cmap="RdBu_r", norm=norm)
        if row == 0:
            ax.set_title(title, fontsize=9, pad=3)
        if col == 0:
            ax.text(
                -0.08,
                0.5,
                metric_name.replace("_", " "),
                transform=ax.transAxes,
                ha="right",
                va="center",
                rotation=90,
                fontsize=10,
                weight="bold",
            )
    cb = fig.colorbar(artist, ax=axes[row, :].tolist(), orientation="vertical", fraction=0.022, pad=0.012)
    cb.set_label(spec["units"])

fig.suptitle("ERA5-Land comparison bias maps", y=0.975, fontsize=13)
savefig(fig, "era5l_energy_flux_bias_maps")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
axes = axes.ravel()
for ax, (metric_name, spec) in zip(axes, COMPARISONS.items()):
    sub = domain_timeseries[domain_timeseries["comparison"].eq(metric_name)].copy()
    sub["time"] = pd.to_datetime(sub["time"])
    for exp, color, lw in [("ERA5L", "0.15", 1.4), ("OL", "#2878B5", 1.0), ("DA", "#F28E2B", 1.0)]:
        ss = sub[sub["experiment"].eq(exp)]
        ax.plot(ss["time"], ss["value"], label="ERA5-Land" if exp == "ERA5L" else exp, color=color, linewidth=lw)
    ax.set_title(metric_name.replace("_", " "), fontsize=10)
    ax.set_ylabel(spec["units"])
    ax.grid(True, alpha=0.25)
axes[0].legend(frameon=False, ncol=3, fontsize=9)
savefig(fig, "era5l_energy_flux_domain_timeseries")
plt.show()

## Save Metrics

The full monthly M36 fields are cached separately by variable. The primary summary product below stores map-level metrics, not all monthly fields, to avoid a very large monolithic NetCDF.

In [ ]:
metric_ds.attrs.update(
    OrderedDict(
        title="ERA5-Land basic energy fluxes versus GEOS-LDAS OL/DA on M36 grid",
        note="ERA5-Land slhf/sshf/e are moda monthly averaged accumulated fields: J m-2 day-1 for heat fluxes and m day-1 for evaporation, sign-flipped to the GEOS-LDAS positive-from-surface convention before comparison. Known bad ERA5-Land monthly averaged accumulated months are handled by drop_era5l_known_bad_accum_months.",
        era5l_documentation=ERA5L_DOC_URL,
        era5l_product=ERA5L_PRODUCT,
        era5l_product_type=ERA5L_PRODUCT_TYPE,
        era5l_stream=ERA5L_STREAM,
        era5l_accumulated_field_conversion="moda accumulated monthly means interpreted as daily accumulations; heat flux J m-2 day-1 / 86400 -> W m-2; evaporation m day-1 * 1000 / 86400 -> kg m-2 s-1; monthly total multiplies daily kg m-2 day-1 by days_in_month",
        era5l_sign_conversion="multiply ERA5-Land slhf/sshf/e by -1 so positive comparison values are upward/from-surface like GEOS-LDAS LHLAND/SHLAND/EVLAND",
        drop_era5l_known_bad_accum_months=str(DROP_ERA5L_KNOWN_BAD_ACCUM_MONTHS),
        era5l_known_bad_accum_months=f"{ERA5L_KNOWN_BAD_ACCUM_START} through {ERA5L_KNOWN_BAD_ACCUM_END}",
        era5l_source=str(ERA5L_ENERGY_NC),
        ol_source=str(OL_FLUX_NC),
        da_source=str(DA_FLUX_NC),
        weights=str(WEIGHTS_PATH),
        min_pairs=MIN_PAIRS,
        periods="; ".join([f"{t0} through {t1}" for t0, t1 in PERIODS]),
        periodized_metric_summary_csv=str(OUTPUT_PERIOD_CSV),
        periodized_m36_summary_nc=str(OUTPUT_PERIOD_NC),
    )
)
period_metric_ds.attrs.update(metric_ds.attrs)
metric_ds.to_netcdf(OUTPUT_NC, mode="w")
period_metric_ds.to_netcdf(OUTPUT_PERIOD_NC, mode="w")
print("saved", OUTPUT_NC)
print("saved", OUTPUT_PERIOD_NC)
print("variables:", len(metric_ds.data_vars), "periodized variables:", len(period_metric_ds.data_vars))
